In [6]:
#!C:\Program Files\Python313\python.exe

"""
PROJECT 2 — Salary Prediction (COMP3800 P02)
Fall 2025
"""

# Import required libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error

# ML algorithms
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge

In [7]:
# Set seed based on course number for reproducibility
np.random.seed(3800)

# Load dataset
df = pd.read_csv("salaries2025.csv")

# Preview the data
df.head()

,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,2025,EN,FT,Data Analist,65664,EUR,69120,NL,0,NL,M
1,2025,EN,FT,Data Analist,47652,EUR,50160,NL,0,NL,M
2,2025,EN,FT,Data Engineer,158113,USD,158113,US,0,US,M
3,2025,EN,FT,Data Engineer,87795,USD,87795,US,0,US,M
4,2025,EX,FT,Data Engineer,351410,USD,351410,US,0,US,M


In [8]:
# Split into training (before 2025) and test set (year 2025 only)
df_train = df[df["work_year"] < 2025]
df_test = df[df["work_year"] == 2025]

# Features and target
X_train = df_train.drop(columns=["salary_in_usd"])
y_train = df_train["salary_in_usd"]

X_test = df_test.drop(columns=["salary_in_usd"])
y_test = df_test["salary_in_usd"]

In [9]:
# Numeric and categorical columns
numeric_features = ["remote_ratio"]
categorical_features = [
    "experience_level", "employment_type", "job_title",
    "employee_residence", "company_location", "company_size"
]

# Preprocess: scale numeric, one-hot-encode categorical
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

In [10]:
# Models to test
models = [
    ("ridge", Ridge()),
    ("gb", GradientBoostingRegressor()),
    ("rf", RandomForestRegressor())
]

# Hyperparameter grids
param_grid = [
    {"model__alpha": [0.1, 1, 10]},  
    {"model__learning_rate": [0.05, 0.1], "model__n_estimators": [100, 200]},
    {"model__n_estimators": [200, 400], "model__max_depth": [5, 10]}
]

# Time-aware cross-validation (no data leakage)
tscv = TimeSeriesSplit(n_splits=5)

# Train all models and collect results
results = []
for (name, model), params in zip(models, param_grid):
    pipe = Pipeline([("prep", preprocessor), ("model", model)])
    g = GridSearchCV(pipe, params, cv=tscv, scoring="neg_mean_absolute_error")
    g.fit(X_train, y_train)
    results.append((name, g.best_params_, -g.best_score_, g.best_estimator_))

# Sort top 10 by best validation MAE
results_df = pd.DataFrame(results, columns=["model", "params", "cv_mae", "estimator"])
results_df = results_df.sort_values(by="cv_mae").head(10)
results_df

,model,params,cv_mae,estimator
0,ridge,{'model__alpha': 10},45802.767758,"(ColumnTransformer(transformers=[('num', Stand..."
1,gb,"{'model__learning_rate': 0.1, 'model__n_estima...",45931.768016,"(ColumnTransformer(transformers=[('num', Stand..."
2,rf,"{'model__max_depth': 10, 'model__n_estimators'...",46688.500111,"(ColumnTransformer(transformers=[('num', Stand..."


In [11]:
# Select the best model
best_model = results_df.iloc[0]["estimator"]

# Train fully on pre-2025 data
best_model.fit(X_train, y_train)

# Predict on 2025 data
preds = best_model.predict(X_test)

# Compute MAE for 2025
mae_2025 = mean_absolute_error(y_test, preds)
mae_2025

47349.20290542288

In [12]:
## Modeling Process

"""
This project predicts salaries using historical data.
Data prior to 2025 is used for training and model tuning.
2025 data is held out and used only once at the end to measure generalization performance.
A time-series cross-validation split prevents future data leakage.
Preprocessing includes one-hot encoding categorical features and scaling numeric features.
Three models were tuned: linear Ridge regression,
Gradient Boosting, and Random Forest. The top 10 configurations are reported.
"""

'\nThis project predicts salaries using historical data.\nData prior to 2025 is used for training and model tuning.\n2025 data is held out and used only once at the end to measure generalization performance.\nA time-series cross-validation split prevents future data leakage.\nPreprocessing includes one-hot encoding categorical features and scaling numeric features.\nThree models were tuned: linear Ridge regression,\nGradient Boosting, and Random Forest. The top 10 configurations are reported.\n'

In [13]:
## Top 10 Model Configurations

"""
(model | params | cv_mae | estimator)
Displayed in the DataFrame above.
"""

'\n(model | params | cv_mae | estimator)\nDisplayed in the DataFrame above.\n'

In [14]:
## Final Assessment for 2025

"""
The top model was adjusted on the entirety of pre-2025 data and assessed with 2025 salary data.
"""


## **MAE Outcome:** *(value from output)*


### Reflection
"""
This result is not surprising to me because salary trends vary by year,
and the models could have underfitted or overfitted to the past data.
Real-world salary data is noisy,
and new roles/markets in 2025 are likely to behave differently than prior to 2025.
"""

'\nThis result is not surprising to me because salary trends vary by year,\nand the models could have underfitted or overfitted to the past data.\nReal-world salary data is noisy,\nand new roles/markets in 2025 are likely to behave differently than prior to 2025.\n'